# Act 1 — The VPC is global. The subnet is regional.

The single most important sentence in GCP networking. If you come from AWS, where a VPC is a regional thing and a subnet is a zonal thing, GCP rearranges both: **VPCs are global** (one VPC spans every region on the planet), and **subnets are regional** (each subnet has an IP range that spans every zone in its region).

This isn't cosmetic. It's why GCP needs *one* VPC to hold a multi-region workload, why you don't manage a subnet-per-AZ for high availability, and why VPC firewall rules apply across regions by default. Once this is internalised, the rest of the chapter follows.

## VPC shape — global, with regional subnets

A **VPC** is a project-scoped, global network resource. Inside one VPC you have:

- **Subnets** — *regional*. Each subnet has a primary IPv4 CIDR (for VM primary IPs) and zero-or-more **secondary CIDRs** (for alias IP ranges — used by GKE pods and services).
- **Routes** — entries in the VPC routing table. Default routes (`0.0.0.0/0` → internet, subnet routes for each subnet) are auto-created; you add custom routes for VPNs, NVAs, or Cloud Router-learned routes.
- **Firewall rules** — VPC-scoped, stateful. Apply to *every* instance in the VPC unless filtered by source/target.

**Two VPC types:**

- **Auto mode** — GCP auto-creates one `/20` subnet per region with a default CIDR layout. Convenient for sandboxes; the CIDRs aren't yours, so it's not suitable for any real enterprise network.
- **Custom mode** — you create subnets explicitly, choose CIDRs, pick regions. The right default for any non-trivial setup.

**Compare:** an AWS VPC is regional; you need one per region (and you peer or use Transit Gateway to connect them). GCP saves you that work — one VPC, subnets in every region you want. Azure VNets are regional like AWS; the GCP global model is the unusual one.

## IP planning — primary, secondary, alias

When you create a subnet you pick a **primary CIDR** — say `10.0.1.0/24` in `us-central1` — which is the pool VM primary IPs come from.

**Secondary CIDRs** are extra IP ranges attached to the same subnet, used for **alias IP ranges**. A VM in this subnet can be given a slice of a secondary range, and individual processes (or containers, or pods) bind to those alias IPs. GKE leans on this hard: pods get IPs from one secondary range, services from another, both routable directly inside the VPC. This is why GKE clusters in VPC-native mode can scale to many thousands of pods without exhausting the primary subnet.

**Plan ranges with peering and hybrid in mind.** Two VPCs you ever want to connect (peering, Cloud VPN, Interconnect) must have non-overlapping CIDRs. The cost of overlap discovered late is renumbering an entire network — months of work. Reserve large parent allocations (`10.0.0.0/8` to GCP, `10.100.0.0/16` to AWS, `10.200.0.0/16` to on-prem) early, before any project owner picks freely.

**Private Google Access** is a per-subnet toggle. When on, VMs *without* external IPs can reach Google APIs (`storage.googleapis.com`, `pubsub.googleapis.com`, etc.) over Google's internal network — no internet gateway, no NAT, no exposure. Always on for private subnets.

# Act 2 — Firewall rules and routes

A VPC has one routing table and one firewall ruleset, both global to that VPC. Two characteristics of GCP firewall rules are worth knowing before you start writing them: they're **stateful**, and they **target by tag or service account**, not by subnet.

## VPC Firewall Rules — stateful, targeted by tag or SA

Every VPC has an implicit firewall:

- **Default-deny ingress** from anywhere.
- **Default-allow egress** to anywhere.

You override with rules. Each rule has:

- **Direction** — `INGRESS` or `EGRESS`.
- **Action** — `allow` or `deny`.
- **Priority** — 0 (highest) to 65535 (lowest). Lower number wins.
- **Source / destination** — IP ranges, source tags, or source service accounts (ingress); destination CIDRs (egress).
- **Targets** — which instances the rule applies to. `target tags`, `target service accounts`, or all instances in the VPC.
- **Protocol/ports** — `tcp:80`, `tcp:443`, `tcp:1024-65535`, etc.

**Stateful.** A reply on an established connection is automatically allowed, regardless of egress rules. You don't write "allow egress 1024-65535 for return traffic" the way you would on AWS NACLs. Same model as AWS Security Groups and Azure NSGs.

**Targeting by tag.** Attach `tags: [web]` to a VM template and write `target-tags=web` on a firewall rule — the rule applies only to VMs with that tag. This is GCP's analogue of AWS Security Group membership, but applied by string tag rather than SG attachment. It scales: one VPC, one rule, applies across regions to every instance with the tag.

**Targeting by service account.** Attach a service account to a VM and write `target-service-accounts=app-sa@…` on a rule. The rule applies only to VMs running as that SA. The benefit over tags: tags are an honour system (anyone with VM-create permission can set them); SAs are gated by IAM (`iam.serviceAccountUser` on the SA). Use SAs for security-critical rules.

**Hierarchical Firewall Policies** (notebook 02) sit *above* VPC firewall rules — they evaluate first, at the Org or folder level, and can deny across every VPC in the descendant projects. Use them for non-negotiable network rules; use VPC rules for per-app shaping.

## Routes and Cloud NAT

**Default routes** in every VPC: `0.0.0.0/0` to the default internet gateway (only matters for VMs with external IPs), plus a route for each subnet.

**Custom routes** point at next hops — another VM (acting as an NVA), a VPN tunnel, or a Cloud Router. They can be priority-ordered the same way firewall rules are.

**Cloud NAT** lets private-IP-only VMs reach the public internet without giving them an external IP. It's a *regional, managed* NAT — no NAT gateway resource to provision (unlike AWS's NAT Gateway), no instance-based NAT to babysit. You attach it to a Cloud Router and it provides outbound NAT for any VM in the specified subnets.

**The pattern:** private subnets host your workload VMs; Cloud NAT gives them outbound internet (for package downloads, third-party API calls); Private Google Access gives them inbound-to-Google-APIs without internet. Together that's the standard private-subnet recipe.

# Act 3 — Connecting two networks inside GCP

Once you have more than one VPC, the question is how they talk. GCP offers three answers, with deliberately different trade-offs: **VPC Peering** (cheap, easy, non-transitive), **Shared VPC** (one VPC, many projects), and **Private Service Connect** (one consumer-facing endpoint, many producers behind it).

## VPC Peering

**VPC Peering** connects two VPCs at the routing level. Each side adds the other's subnets to its route table; instances on either side can talk to each other by RFC1918 IP, as if on the same network.

Three limits worth knowing:

- **Non-transitive.** If A peers with B and B peers with C, A *cannot* reach C through B. There is no chained transit. Use a hub-and-spoke topology with Cloud VPN/Interconnect-as-hub if you need transit.
- **CIDR ranges must not overlap.** Mentioned in Act 1, repeated here because peering is where overlap bites first.
- **Firewall rules don't cross peerings.** Each side's VPC firewall rules apply independently. Inbound traffic from a peered VPC is evaluated against your VPC's ingress rules as if it came from any other source.

Good for: connecting a Shared VPC host to peer-specific consumer VPCs; connecting environment VPCs (dev↔shared-services) where transit isn't needed.

## Shared VPC — the GCP-unique multi-project pattern

**Shared VPC** lets multiple projects use a single VPC owned by a designated **host project**. The pattern:

- One **host project** owns the VPC, subnets, firewall rules, routes, VPNs, Interconnects.
- N **service projects** attach to the host project. Their workloads (VMs, GKE clusters, Cloud SQL with PSC, etc.) live in the host's subnets but bill to the service project.

**Why this is the recommended enterprise pattern:**

- Central network team owns network configuration in the host project. App teams own only their compute, in their own service projects.
- Per-subnet IAM — `roles/compute.networkUser` on `subnet-app-prod-us-central1` only — lets you give a team rights to *deploy into a subnet* without giving them rights to *change the VPC*.
- All workloads share IP space, so cross-team service-to-service traffic stays inside the VPC with no peering.

**Compare:** AWS handles this with Resource Access Manager (sharing a subnet) — Shared VPC is the more cohesive equivalent. Azure has VNet peering as the primary pattern; the cross-subscription "shared" story is weaker.

Notebook 04's GKE clusters and Cloud Run services typically run in Shared VPC service projects in production environments.

## Private Service Connect

**Private Service Connect (PSC)** is the modern way to expose a service in one VPC (the **producer**) to consumers in other VPCs or on-prem, *without* peering and *without* IP overlap considerations.

The shape:

- The **producer** publishes a service behind an internal load balancer with a **Service Attachment**.
- The **consumer** creates a **PSC endpoint** in their own VPC — a forwarding rule with an IP from *their* address space.
- Traffic from the consumer to that endpoint IP is routed (via Google's backbone) to the producer's LB.

Three common shapes:

- **PSC for Google services** — reach `storage.googleapis.com`, `bigquery.googleapis.com`, etc. via a private IP in your VPC. The successor to the older Private Service Access for Google APIs.
- **PSC for managed services** — Cloud SQL, AlloyDB, Memorystore now publish PSC endpoints. The producer's network is invisible to you.
- **PSC for third-party / SaaS** — Datadog, Snowflake, MongoDB Atlas, Confluent all publish PSC endpoints. The consumer reaches them via internal IP, no public internet.

**Why this matters.** Before PSC, connecting to a SaaS or managed service often meant VPC peering (with all its CIDR and transit constraints) or going over the internet. PSC is the equivalent of AWS PrivateLink or Azure Private Endpoint — and arguably more flexible because the consumer endpoint's IP is fully under the consumer's control.

# Act 4 — Reaching outside GCP

Three mechanisms get traffic between your GCP VPC and on-prem (or another cloud). Pick on bandwidth and durability requirements: **Cloud VPN** is fast to stand up; **Dedicated Interconnect** gives you a private fiber drop; **Partner Interconnect** is the middle ground.

## Cloud VPN — HA VPN vs Classic

IPsec VPN over the public internet. Two flavours:

- **HA VPN** — two tunnels on two interfaces with two public IPs, BGP via Cloud Router. **99.99% availability SLA.** The default for new deployments.
- **Classic VPN** — single-interface, single-tunnel. Static or BGP routing. **99.9% SLA.** Avoid for new work; the SLA difference is meaningful at scale.

Throughput per tunnel: ~3 Gbps. Practical bandwidth needs above that argue for Interconnect.

**Cloud Router** is the BGP speaker on GCP's side. Every HA VPN and every Interconnect attachment is paired with a Cloud Router that exchanges routes with your on-prem peer via BGP. The route exchange means your on-prem CIDRs appear in the VPC's routing table dynamically — no manual route additions when on-prem networks change.

## Dedicated and Partner Interconnect

**Dedicated Interconnect** is a physical fiber circuit from a Google PoP (peering location) directly into your network gear. 10 Gbps or 100 Gbps per circuit; up to 8 circuits per attachment (link aggregation). You build out — Google provides the cross-connect, you arrange your own fiber and BGP. Best price-per-gigabit at scale, longest setup time (weeks to months for cross-connect provisioning).

**Partner Interconnect** uses a service provider (Equinix, Megaport, Telefonica, ~40 others) as the intermediary. Speeds from 50 Mbps to 50 Gbps in defined tiers. Faster to stand up, since you only need a relationship with the partner. Slightly higher per-gigabit cost than Dedicated.

**Cross-Cloud Interconnect** is the newest variant — a Google-managed circuit to AWS, Azure, or Oracle Cloud, used for cloud-to-cloud private connectivity without going over the internet. Useful for multi-cloud architectures that need predictable bandwidth.

| Option | Setup time | Bandwidth | SLA | Best for |
|---|---|---|---|---|
| **HA VPN** | Hours | 3 Gbps per tunnel | 99.99% | Modest hybrid, fast bring-up |
| **Partner Interconnect** | Days | 50 Mbps – 50 Gbps | 99.9% / 99.99% (HA) | Mid-scale hybrid |
| **Dedicated Interconnect** | Weeks | 10 / 100 Gbps | 99.9% / 99.99% (HA) | High-bandwidth, cost-sensitive |
| **Cross-Cloud Interconnect** | Days–weeks | 10 / 100 Gbps | 99.9% / 99.99% (HA) | Multi-cloud backbone |

## A note on Private Service Access (legacy)

Before PSC, GCP managed services like Cloud SQL used **Private Service Access** — a peered VPC owned by Google holding the service, attached to your VPC via an automated peering connection over a CIDR range you allocated. It still works and is still used for Cloud SQL, Memorystore, and others by default.

For new designs, prefer PSC where available. The CIDR allocation step in Private Service Access is a recurring source of pain (you reserve a `/16` you can never use elsewhere); PSC sidesteps that with one IP per endpoint.

## What carries into later chapters

Networking is the substrate every other notebook sits on. Cloud Run wires into VPC via Direct VPC Egress (notebook 04, recapped here). GKE relies on VPC-native networking and alias IPs (notebook 04). Cloud SQL uses Private Service Access or PSC for private connectivity (notebook 08). BigQuery and GCS are reachable via Private Google Access from VMs without public IPs (this notebook, throughout).

The three habits to carry forward:

- **Plan CIDRs centrally and early.** Overlap is the most expensive networking mistake to discover late.
- **Shared VPC is the right enterprise default.** Per-project VPCs scattered across the org accumulate connectivity debt fast.
- **Private Service Connect over VPC Peering for managed services and SaaS.** PSC scales; peering does not.

Notebook 07 is traffic flow — load balancers, Cloud Armor, Cloud DNS, Cloud CDN — and how Internet traffic finds your backends through the Google Front End.